## 1. Data Loading & Strict Anti-Leakage (Cut-off Implementation)
To simulate a real-world predictive environment and prevent data leakage, a strict `CUT_OFF_DAY` is established.
*   **Logic:** `CUT_OFF_DAY = MAX_DAY - 60`
*   **Transaction Filter:** `obs_txns = transactions[DAY < CUT_OFF_DAY]`. (Strictly `<` to ensure the cut-off day's transactions do not overlap with the churn label window).
*   **Causal Filter:** Filtered using `WEEK_NO <= cutoff_week` to prevent future promotion leakage.

In [1]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from imblearn.combine import SMOTETomek
from IPython.display import display

# ==============================================================================
# STEP 1: LOAD DATA & SET CUT-OFF (AIRTIGHT ANTI-LEAKAGE)
# ==============================================================================
DATA_PROCESSED = '../Data/Processed/'
DATA_RAW = '../Data/Raw/'
MODELS_DIR = '../models/'
os.makedirs(MODELS_DIR, exist_ok=True)

print("Loading Data...")
transactions = pd.read_parquet(os.path.join(DATA_PROCESSED, 'transactions_master.parquet'), engine='pyarrow')
customer = pd.read_parquet(os.path.join(DATA_PROCESSED, 'customer_base_labeled.parquet'), engine='pyarrow')
demographics = pd.read_parquet(os.path.join(DATA_PROCESSED, 'demographics_imputed.parquet'), engine='pyarrow')
product = pd.read_parquet(os.path.join(DATA_RAW, 'product.parquet'), engine='pyarrow')
causal_data = pd.read_parquet(os.path.join(DATA_RAW, 'causal_data.parquet'), engine='pyarrow')
campaign_table = pd.read_parquet(os.path.join(DATA_PROCESSED, 'campaign_table_clean.parquet'), engine='pyarrow')
campaign_desc = pd.read_parquet(os.path.join(DATA_PROCESSED, 'campaign_desc_clean.parquet'), engine='pyarrow')

# Data sample
print("\nSample data from transactions_master:")
display(transactions.head())

print("\nSample data from causal_data:")
display(causal_data.head())


print("\nSample data from customer_master:")
display(customer.head())

print("\nSample data from demography:")
display(demographics.head())

# SET CUT-OFF
CUT_OFF_DAY = 711 - 60 
print(f"Cut-off Date: Day {CUT_OFF_DAY}")

# 1.1 Filter transactions (create obs_txns - source for all features)
obs_txns = transactions[transactions['DAY'] < CUT_OFF_DAY].copy()

# 1.2 Filter causal data (fix causal leakage)
# Convert DAY to WEEK_NO (assume 1 week = 7 days)
cutoff_week = int(np.ceil(CUT_OFF_DAY / 7))
causal_obs = causal_data[causal_data['WEEK_NO'] <= cutoff_week].copy()


# 1.3 Initialize final table (keep only ID and churn label, no extra fields)
df_final = customer[['household_key', 'is_churn']].copy()
df_final.rename(columns={'is_churn': 'churn_flag'}, inplace=True)
df_final['household_key'] = df_final['household_key'].astype(str)

Loading Data...

Sample data from transactions_master:


,household_key,BASKET_ID,DAY,PRODUCT_ID,QUANTITY,SALES_VALUE,STORE_ID,RETAIL_DISC,TRANS_TIME,WEEK_NO,COUPON_DISC,COUPON_MATCH_DISC,DEPARTMENT,BRAND,COMMODITY_DESC
0,2375,26984851472,1,1004906,1,1.39,364,-0.60,1631,1,0.0,0.0,PRODUCE,Private,POTATOES
1,2375,26984851472,1,1033142,1,0.82,364,0.00,1631,1,0.0,0.0,PRODUCE,National,ONIONS
2,2375,26984851472,1,1036325,1,0.99,364,-0.30,1631,1,0.0,0.0,PRODUCE,Private,VEGETABLES - ALL OTHERS
3,2375,26984851472,1,1082185,1,1.21,364,0.00,1631,1,0.0,0.0,PRODUCE,National,TROPICAL FRUIT
4,2375,26984851472,1,8160430,1,1.50,364,-0.39,1631,1,0.0,0.0,PRODUCE,Private,ORGANICS FRUIT & VEGETABLES



Sample data from causal_data:


,PRODUCT_ID,STORE_ID,WEEK_NO,display,mailer
0,26190,286,70,0,A
1,26190,288,70,0,A
2,26190,289,70,0,A
3,26190,292,70,0,A
4,26190,293,70,0,A



Sample data from customer_master:


,household_key,mean_IPT,std_IPT,last_purchase_day,personalized_threshold,recency,is_churn,Frequency,Monetary,AGE_DESC,MARITAL_STATUS_CODE,INCOME_DESC,HOMEOWNER_DESC,HH_COMP_DESC,HOUSEHOLD_SIZE_DESC,KID_CATEGORY_DESC
0,1,8.506494,4.581494,706,17.669482,5,0,83,4310.160156,65+,A,35-49K,Homeowner,2 Adults No Kids,2,None/Unknown
1,10,142.750000,159.639959,685,462.029919,26,0,9,234.339996,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown
2,100,32.619048,24.499951,691,81.618950,20,0,23,1959.219971,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown
3,1000,5.256000,5.900935,706,17.057870,5,0,130,3972.439941,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown
4,1001,8.897436,13.774237,710,36.445910,1,0,90,4074.020020,45-54,U,50-74K,Homeowner,Unknown,1,None/Unknown



Sample data from demography:


,AGE_DESC,MARITAL_STATUS_CODE,INCOME_DESC,HOMEOWNER_DESC,HH_COMP_DESC,HOUSEHOLD_SIZE_DESC,KID_CATEGORY_DESC,household_key
0,65+,A,35-49K,Homeowner,2 Adults No Kids,2,None/Unknown,1
1,45-54,A,50-74K,Homeowner,2 Adults No Kids,2,None/Unknown,7
2,25-34,U,25-34K,Unknown,2 Adults Kids,3,1,8
3,25-34,U,75-99K,Homeowner,2 Adults Kids,4,2,13
4,45-54,B,50-74K,Homeowner,Single Female,1,None/Unknown,16


Cut-off Date: Day 651


## 2. RFM (Recency-Safe), Stability & Basket Behavior
Calculates core purchasing habits based purely on the observation window.

| Feature | Logic / Formula | Business Insight |
| :--- | :--- | :--- |
| **Frequency** | Unique `BASKET_ID` count. | Customer's overall store visit frequency. |
| **Monetary** | Sum of `SALES_VALUE`. | Total customer lifetime value (historic). |
| **Avg_Items_Per_Basket** | `Total_Items / Frequency` | Basket depth; indicates if the customer does bulk shopping or quick trips. |
| **Basket_Value_Std** | Standard deviation of basket values. | Measures the consistency of the customer's spending per trip. |
| **Recency_Capped** | `clip(CUT_OFF_DAY - last_day, 0, 90)` | Safe recency metric. Capped at 90 days to reduce noise from long-dormant customers. |
| **Inactive_Days_Ratio** | `Recency_Capped / Customer_Lifetime` | The proportion of the customer's lifespan spent inactive. |
| **active_weeks_ratio** | `Active_Weeks / (Lifetime_in_weeks + 1)` | Engagement depth. Clipped to 1.0 to prevent inflating ratios for new customers. |
| **coupon_dependency** | `Total_Coupon_Discount / Monetary` | Price sensitivity; identifies bargain hunters. |
| **promo_usage_ratio** | `Trips_With_Coupon / Frequency` | Frequency of utilizing promotions during visits. |
| **IPT_mean / IPT_std** | Mean and Std of Inter-Purchase Time (days). | Calculates the average gap between visits and its variance. NaNs are handled to avoid "perfect stability" bias. |
| **IPT_CV** | `IPT_std / IPT_mean` | **Crucial habit metric**. Coefficient of Variation of visit gaps. High CV = erratic shopper; Low CV = creature of habit. |

In [2]:
# ==============================================================================
# STEP 2: RFM (RECENCY-SAFE), STABILITY (IPT) & BASKET BEHAVIOR
# ==============================================================================
print("Processing RFM, Stability (IPT), and Behavior...")
obs_txns['household_key'] = obs_txns['household_key'].astype(str)

# 2.1 Basic RFM & basket behavior
agg_funcs = {
    'BASKET_ID': ['nunique', 'count'], # nunique = trips, count = total items
    'SALES_VALUE': ['sum', 'std'],     # sum = Monetary, std = basket value std
    'COUPON_DISC': lambda x: np.sum(np.abs(x)), 
    'DAY': ['min', 'max']  
}

cust_feats = obs_txns.groupby('household_key').agg(agg_funcs).reset_index()
cust_feats.columns = ['household_key', 'Frequency', 'Total_Items', 'Monetary', 'Basket_Value_Std', 'Total_Coupon_Discount', 'first_day', 'last_day']

# -------------------------------------------------------------------------
# FIX HERE: move customer_lifetime calculation above
# -------------------------------------------------------------------------
# 2.2 Recency (recency-safe) & lifetime
cust_feats['customer_lifetime'] = CUT_OFF_DAY - cust_feats['first_day']
cust_feats['customer_lifetime'] = np.where(cust_feats['customer_lifetime'] <= 0, 1, cust_feats['customer_lifetime'])

# Capped recency: from last purchase day (in obs_txns) to cut-off
raw_recency = CUT_OFF_DAY - cust_feats['last_day']
# Cap recency at 90 days (reduce noise from very dormant customers)
cust_feats['Recency_Capped'] = np.clip(raw_recency, 0, 90)

# Inactive streak / purchase gap ratio
cust_feats['Inactive_Days_Ratio'] = raw_recency / cust_feats['customer_lifetime']

# -------------------------------------------------------------------------
# CONTINUE OTHER FEATURES
# -------------------------------------------------------------------------
# Active weeks (store visits) / total weeks in customer lifetime
active_weeks = obs_txns.groupby('household_key')['WEEK_NO'].nunique().reset_index(name='Active_Weeks')
cust_feats = pd.merge(cust_feats, active_weeks, on='household_key', how='left')
cust_feats['active_weeks_ratio'] = np.clip(cust_feats['Active_Weeks'] / ((cust_feats['customer_lifetime'] / 7) + 1), 0, 1)

# Trips with coupon / total trips
txns_with_coupon = obs_txns[obs_txns['COUPON_DISC'] < 0].groupby('household_key')['BASKET_ID'].nunique().reset_index(name='Trips_With_Coupon')
cust_feats = pd.merge(cust_feats, txns_with_coupon, on='household_key', how='left').fillna(0)
cust_feats['promo_usage_ratio'] = cust_feats['Trips_With_Coupon'] / cust_feats['Frequency']

# 2.3 Basket & promotion sensitivity
cust_feats['Avg_Items_Per_Basket'] = cust_feats['Total_Items'] / cust_feats['Frequency']
cust_feats['Discount_Per_Transaction'] = cust_feats['Total_Coupon_Discount'] / cust_feats['Frequency']

# Drop unused columns
cust_feats.drop(columns=['Total_Items', 'Total_Coupon_Discount', 'first_day', 'last_day', 'Active_Weeks', 'Trips_With_Coupon'], inplace=True)
cust_feats.fillna(0, inplace=True)

# 2.4 STABILITY / HABIT (IPT - STRONG SIGNAL)
# Get unique purchase days per customer
unique_days = obs_txns[['household_key', 'DAY']].drop_duplicates().sort_values(['household_key', 'DAY'])
unique_days['Prev_DAY'] = unique_days.groupby('household_key')['DAY'].shift(1)
unique_days['IPT'] = unique_days['DAY'] - unique_days['Prev_DAY']

ipt_stats = unique_days.groupby('household_key').agg(
    IPT_mean=('IPT', 'mean'),
    IPT_std=('IPT', 'std')
).reset_index()

# Fix: filling std with 0 can be misleading; use mean for missing std
ipt_stats['IPT_std'] = ipt_stats['IPT_std'].fillna(ipt_stats['IPT_mean'])

# If mean is also NaN (only 1 purchase), fill with customer lifetime
cust_life_map = dict(zip(cust_feats['household_key'], cust_feats['customer_lifetime']))
ipt_stats['IPT_mean'] = ipt_stats['IPT_mean'].fillna(ipt_stats['household_key'].map(cust_life_map))
ipt_stats['IPT_std'] = ipt_stats['IPT_std'].fillna(ipt_stats['household_key'].map(cust_life_map))

# Coefficient of Variation (CV) of IPT
ipt_stats['IPT_CV'] = ipt_stats['IPT_std'] / (ipt_stats['IPT_mean'] + 1e-5)
ipt_stats.fillna(0, inplace=True) # Fill remaining rare cases with 0

# Merge into cust_feats
cust_feats = pd.merge(cust_feats, ipt_stats, on='household_key', how='left').fillna(0)

Processing RFM, Stability (IPT), and Behavior...


## 3. Marketing Responsiveness, Brand Affinity & Rolling Trend
Captures how the customer interacts with store environments, brands, and recent momentum.

| Feature | Logic / Formula | Business Insight |
| :--- | :--- | :--- |
| **Private_Brand_Ratio** | `(Private + 1) / (Private + National + 2)` | Brand loyalty using **Laplace Smoothing** to prevent NaN bias for single-item buyers. High ratio = high switching cost. |
| **Display / Mailer_Responsiveness** | `Qty_from_Promo / Total_Qty` | Measures how effectively in-store displays and mailers drive the customer's volume. |
| **Rolling_Freq_Slope** | `np.polyfit` over M1, M2, M3 (30-day windows). | Linear regression slope of recent shopping frequency. Captures momentum (cooling down vs. warming up). |
| **Primary_Store_ID** | `mode()` of `STORE_ID` | Identifies the store the customer visits most frequently. Highly useful for the Modeling team (M5) to evaluate churn risk at the specific store level. |

In [3]:
# ==============================================================================
# STEP 3: MARKETING RESPONSIVENESS, BRAND AFFINITY & ROLLING TREND
# ==============================================================================
print("Processing Brand, Causal, and Rolling Trend...")

# 3.1 Brand affinity (private ratio)
if 'BRAND' not in obs_txns.columns:
    obs_txns = pd.merge(obs_txns, product[['PRODUCT_ID', 'BRAND']], on='PRODUCT_ID', how='left')

brand_qty = obs_txns.groupby(['household_key', 'BRAND'])['QUANTITY'].sum().unstack(fill_value=0).reset_index()
brand_qty['Private_Brand_Ratio'] = (brand_qty.get('Private', 0) + 1) / (brand_qty.get('Private', 0) + brand_qty.get('National', 0) + 2)

# 3.2 Causal data (causal leakage fixed)
causal_obs['PRODUCT_ID'] = causal_obs['PRODUCT_ID'].astype(str)
causal_obs['WEEK_NO'] = causal_obs['WEEK_NO'].astype(int)
obs_txns['PRODUCT_ID'] = obs_txns['PRODUCT_ID'].astype(str)
obs_txns['WEEK_NO'] = obs_txns['WEEK_NO'].astype(int)

causal_obs['is_display'] = np.where(~causal_obs['display'].isin(['0', ' ', 'None', 'NaN']), 1, 0)
causal_obs['is_mailer'] = np.where(~causal_obs['mailer'].isin(['0', ' ', 'None', 'NaN']), 1, 0)
causal_agg = causal_obs.groupby(['WEEK_NO', 'PRODUCT_ID']).agg({'is_display': 'max', 'is_mailer': 'max'}).reset_index()

txn_causal = pd.merge(obs_txns[['household_key', 'WEEK_NO', 'PRODUCT_ID', 'QUANTITY']], causal_agg, on=['WEEK_NO', 'PRODUCT_ID'], how='left').fillna(0)
txn_causal['Qty_Display'] = txn_causal['QUANTITY'] * txn_causal['is_display']
txn_causal['Qty_Mailer'] = txn_causal['QUANTITY'] * txn_causal['is_mailer']

causal_features = txn_causal.groupby('household_key').agg(
    Total_Qty=('QUANTITY', 'sum'),
    Qty_Display=('Qty_Display', 'sum'),
    Qty_Mailer=('Qty_Mailer', 'sum')
).reset_index()

causal_features['Display_Responsiveness'] = causal_features['Qty_Display'] / (causal_features['Total_Qty'] + 1e-5)
causal_features['Mailer_Responsiveness'] = causal_features['Qty_Mailer'] / (causal_features['Total_Qty'] + 1e-5)
causal_features.drop(columns=['Total_Qty', 'Qty_Display', 'Qty_Mailer'], inplace=True)

# 3.3 Rolling trend (linear regression slope over last 3 months)
# Split last 90 days into 3 periods (each 30 days)
obs_txns['days_to_cutoff'] = CUT_OFF_DAY - obs_txns['DAY']

# Monthly frequency
m1 = obs_txns[obs_txns['days_to_cutoff'] <= 30].groupby('household_key')['BASKET_ID'].nunique().reset_index(name='M1')
m2 = obs_txns[(obs_txns['days_to_cutoff'] > 30) & (obs_txns['days_to_cutoff'] <= 60)].groupby('household_key')['BASKET_ID'].nunique().reset_index(name='M2')
m3 = obs_txns[(obs_txns['days_to_cutoff'] > 60) & (obs_txns['days_to_cutoff'] <= 90)].groupby('household_key')['BASKET_ID'].nunique().reset_index(name='M3')

trend_df = pd.DataFrame({'household_key': obs_txns['household_key'].unique()})
trend_df = trend_df.merge(m1, on='household_key', how='left').merge(m2, on='household_key', how='left').merge(m3, on='household_key', how='left').fillna(0)

# Linear slope over 3 points (M3 -> M2 -> M1)
# Simple slope: (y3 - y1) / (x3 - x1). Here x is time, so slope = (M1 - M3) / 2
def calc_slope(row):
    return np.polyfit([1, 2, 3], [row['M3'], row['M2'], row['M1']], 1)[0]

trend_df['Rolling_Freq_Slope'] = trend_df.apply(calc_slope, axis=1)
trend_df = trend_df[['household_key', 'Rolling_Freq_Slope']]

# -------------------------------------------------------------------------
# ADDITION: 3.4 Primary Store Affinity
# -------------------------------------------------------------------------
# Find the most frequently visited store (mode) for each household
primary_store = obs_txns.groupby('household_key')['STORE_ID'].agg(
    lambda x: pd.Series.mode(x)[0] if not x.mode().empty else np.nan
).reset_index(name='Primary_Store_ID')

primary_store['Primary_Store_ID'] = primary_store['Primary_Store_ID'].astype(str)

Processing Brand, Causal, and Rolling Trend...


## 4. Campaign Exposure & Demographics
Measures the impact of direct marketing campaigns and profile completeness.

| Feature | Logic / Formula | Business Insight |
| :--- | :--- | :--- |
| **Total_Campaigns_Received** | Count of campaigns before CUT_OFF_DAY. | Overall marketing investment in this customer. |
| **Camp_Count_TypeA/B/C** | Crosstab of campaign types. | Identifies which campaign formats the customer is exposed to. |
| **Campaigns_Last_30D** | Campaigns received in the last 30 days. | **Campaign Fatigue**. Measures if the customer is being spammed right before the churn window. |
| **Days_Since_Last_Camp** | `CUT_OFF_DAY - last_campaign_day`. | Evaluates if the customer is feeling neglected by marketing. (Un-targeted customers imputed with max distance). |
| **has_demographic_info** | `1` if age info exists, else `0`. | Proxy for membership program engagement/completeness. |

In [4]:
# ==============================================================================
# STEP 4: CAMPAIGN EXPOSURE & HAS_DEMOGRAPHICS
# ==============================================================================
print("Processing Campaign and Demographics...")

# 4.1 Campaign exposure
if 'DESCRIPTION' in campaign_table.columns:
    campaign_table = campaign_table.drop(columns=['DESCRIPTION'])
camp_full = pd.merge(campaign_table, campaign_desc, on='CAMPAIGN', how='left')

# Filter campaigns before cut-off
camp_obs = camp_full[camp_full['START_DAY'] <= CUT_OFF_DAY].copy()
camp_obs['household_key'] = camp_obs['household_key'].astype(str)

# Campaign exposure (total)
camp_counts = camp_obs.groupby('household_key').size().reset_index(name='Total_Campaigns_Received')

# -------------------------------------------------------------------------
# ADDITION: Categorize campaigns by type (Type A, B, C)
# -------------------------------------------------------------------------
camp_types = camp_obs.groupby(['household_key', 'DESCRIPTION']).size().unstack(fill_value=0).reset_index()

# Standardize column names (e.g., Camp_Count_TypeA)
camp_type_cols = {col: f'Camp_Count_{col}' for col in camp_types.columns if col != 'household_key'}
camp_types.rename(columns=camp_type_cols, inplace=True)
# -------------------------------------------------------------------------

# Campaign fatigue (too many in the last 30 days)
camp_obs['days_to_cutoff_camp'] = CUT_OFF_DAY - camp_obs['START_DAY']
fatigue = camp_obs[camp_obs['days_to_cutoff_camp'] <= 30].groupby('household_key').size().reset_index(name='Campaigns_Last_30D')

# Campaign recency
camp_recency = camp_obs.groupby('household_key')['START_DAY'].max().reset_index()
camp_recency['Days_Since_Last_Camp'] = CUT_OFF_DAY - camp_recency['START_DAY']

# Assemble Campaign features
campaign_features = pd.merge(camp_counts, fatigue, on='household_key', how='outer').fillna(0)
campaign_features = pd.merge(campaign_features, camp_recency[['household_key', 'Days_Since_Last_Camp']], on='household_key', how='outer')

# Merge Campaign Types (A, B, C) into campaign_features
campaign_features = pd.merge(campaign_features, camp_types, on='household_key', how='outer').fillna(0)

# 4.2 Demographics (has_demo)
demographics['household_key'] = demographics['household_key'].astype(str)
demo_feature = pd.DataFrame({'household_key': obs_txns['household_key'].unique()})
demo_feature = pd.merge(demo_feature, demographics[['household_key', 'AGE_DESC']], on='household_key', how='left')
demo_feature['has_demographic_info'] = np.where(demo_feature['AGE_DESC'].notna(), 1, 0)
demo_feature.drop(columns=['AGE_DESC'], inplace=True)

Processing Campaign and Demographics...


In [5]:
# ==============================================================================
# STEP 5 (AIRTIGHT): ASSEMBLY, SCALING & SMOTETomek
# ==============================================================================
print("Assembling final dataset...")

# 5.1 Merge all feature tables
df_final = pd.merge(df_final, cust_feats, on='household_key', how='inner')
df_final = pd.merge(df_final, brand_qty[['household_key', 'Private_Brand_Ratio']], on='household_key', how='left')
df_final = pd.merge(df_final, causal_features, on='household_key', how='left')
df_final = pd.merge(df_final, trend_df, on='household_key', how='left')
df_final = pd.merge(df_final, campaign_features, on='household_key', how='left')
df_final = pd.merge(df_final, demo_feature, on='household_key', how='left')

# ADDITION: Merge Primary Store
df_final = pd.merge(df_final, primary_store, on='household_key', how='left')

# Handle NaN values
df_final['Days_Since_Last_Camp'].fillna(CUT_OFF_DAY, inplace=True)
df_final['Primary_Store_ID'].fillna('Unknown', inplace=True)
df_final.fillna(0, inplace=True)

# 5.2 EXPORT RAW DATA (FOR M2 CLUSTERING AND EDA)
df_final.to_csv(os.path.join(MODELS_DIR, 'final_ML_features.csv'), index=False)
print(f"Final DataFrame shape: {df_final.shape}")



Assembling final dataset...
Final DataFrame shape: (2499, 27)


In [6]:
# ==============================================================================
# STEP 6: EXPORT TREATMENT FLAGS FOR PROPENSITY SCORE MATCHING (PSM)
# ==============================================================================
import os
print("Generating Treatment Flags for PSM (Method C: Intensity-based)...")

# Tạo thư mục nếu chưa tồn tại
PSM_DIR = '../models/psm_inputs/'
os.makedirs(PSM_DIR, exist_ok=True)

# 1. Khởi tạo DataFrame tổng hợp từ tập khách hàng hiện tại
psm_flags = pd.DataFrame({'household_key': df_final['household_key'].unique()})

# 2. Lấy thông tin promo_usage_ratio từ bảng cust_feats đã tính ở Step 2
# (Đảm bảo cust_feats đã tồn tại trong bộ nhớ)
psm_flags = pd.merge(psm_flags, cust_feats[['household_key', 'promo_usage_ratio']], on='household_key', how='left')

# Xử lý NaN (nếu có khách hàng không có giao dịch nào)
psm_flags['promo_usage_ratio'].fillna(0, inplace=True)

# 3. Tính toán ngưỡng trung bình của toàn bộ tập khách hàng
mean_promo_ratio = psm_flags['promo_usage_ratio'].mean()
print(f"-> Threshold (Mean Promo Ratio): {mean_promo_ratio:.4f}")

# 4. Gắn cờ is_treated = 1 nếu tỷ lệ sử dụng khuyến mãi lớn hơn mức trung bình
psm_flags['is_treated'] = (psm_flags['promo_usage_ratio'] > mean_promo_ratio).astype(int)

# 5. Cập nhật treatment_source để phản ánh đúng Phương án C, đồng thời giữ chuẩn Schema cho M5
psm_flags['treatment_source'] = psm_flags['is_treated'].apply(
    lambda x: 'Heavy_Promo_User' if x == 1 else 'Light/No_Promo_User'
)

# 6. Lưu lại mốc thời gian cut-off để chống Data Leakage
psm_flags['treatment_cutoff_day'] = CUT_OFF_DAY

# Lọc lại Schema chuẩn theo đúng quy ước của nhóm
psm_export_df = psm_flags[['household_key', 'is_treated', 'treatment_source', 'treatment_cutoff_day']]

# 7. Export ra CSV
psm_csv_path = os.path.join(PSM_DIR, 'psm_treatment_flags.csv')
psm_export_df.to_csv(psm_csv_path, index=False)

print(f"\nHoàn thành! Đã xuất {len(psm_export_df)} dòng dữ liệu treatment flag.")
print(f"Phân phối Treatment (is_treated):")
print(psm_export_df['is_treated'].value_counts(normalize=True) * 100)
print(f"File lưu tại: {psm_csv_path}")

Generating Treatment Flags for PSM (Method C: Intensity-based)...
-> Threshold (Mean Promo Ratio): 0.0536



Hoàn thành! Đã xuất 2499 dòng dữ liệu treatment flag.
Phân phối Treatment (is_treated):
is_treated
0    68.987595
1    31.012405
Name: proportion, dtype: float64
File lưu tại: ../models/psm_inputs/psm_treatment_flags.csv


In [7]:
# ==============================================================================
# TEST PHƯƠNG ÁN A: CHỈ DÙNG COUPON (HÀNH VI CHỦ ĐÍCH)
# ==============================================================================
# Lọc khách hàng có phát sinh giao dịch Coupon
coupon_hhs_A = set(obs_txns[obs_txns['COUPON_DISC'] < 0]['household_key'])

# Dùng biến tạm để test, không add vào df_final
temp_treated_A = df_final['household_key'].apply(lambda x: 1 if x in coupon_hhs_A else 0)

print("--- KẾT QUẢ PHƯƠNG ÁN A (Chỉ Coupon) ---")
print(temp_treated_A.value_counts(normalize=True) * 100)

--- KẾT QUẢ PHƯƠNG ÁN A (Chỉ Coupon) ---
household_key
1    68.307323
0    31.692677
Name: proportion, dtype: float64


In [8]:
# ==============================================================================
# TEST PHƯƠNG ÁN B: COUPON + MAILER (TRONG 90 NGÀY GẦN NHẤT)
# ==============================================================================
# 1. Lọc giao dịch trong 90 ngày cuối
recent_txns = obs_txns[obs_txns['DAY'] >= (CUT_OFF_DAY - 90)]

# 2. Lọc Causal theo tuần tương ứng (90 ngày ~ 13 tuần)
recent_cutoff_week = int(np.ceil((CUT_OFF_DAY - 90) / 7))
recent_txn_causal = txn_causal[txn_causal['WEEK_NO'] >= recent_cutoff_week]

# 3. Lấy tập khách hàng
recent_coupon_hhs = set(recent_txns[recent_txns['COUPON_DISC'] < 0]['household_key'])
recent_mailer_hhs = set(recent_txn_causal[recent_txn_causal['Qty_Mailer'] > 0]['household_key'])

# 4. Dùng biến tạm để test
temp_treated_B = df_final['household_key'].apply(
    lambda x: 1 if (x in recent_coupon_hhs) or (x in recent_mailer_hhs) else 0
)

print("--- KẾT QUẢ PHƯƠNG ÁN B (90 ngày gần nhất) ---")
print(temp_treated_B.value_counts(normalize=True) * 100)

--- KẾT QUẢ PHƯƠNG ÁN B (90 ngày gần nhất) ---
household_key
1    87.314926
0    12.685074
Name: proportion, dtype: float64


In [9]:
# ==============================================================================
# TEST PHƯƠNG ÁN C: CƯỜNG ĐỘ SỬ DỤNG (HEAVY VS LIGHT PROMO USERS)
# ==============================================================================
# Tính mức trung bình của toàn tập khách hàng
mean_promo_ratio = cust_feats['promo_usage_ratio'].mean()

# Lọc những người dùng nhiều hơn mức trung bình
heavy_promo_hhs = set(cust_feats[cust_feats['promo_usage_ratio'] > mean_promo_ratio]['household_key'])

# Dùng biến tạm để test
temp_treated_C = df_final['household_key'].apply(lambda x: 1 if x in heavy_promo_hhs else 0)

print(f"--- KẾT QUẢ PHƯƠNG ÁN C (promo_usage_ratio > {mean_promo_ratio:.4f}) ---")
print(temp_treated_C.value_counts(normalize=True) * 100)

--- KẾT QUẢ PHƯƠNG ÁN C (promo_usage_ratio > 0.0536) ---
household_key
0    68.987595
1    31.012405
Name: proportion, dtype: float64


In [10]:
# ==============================================================================
# TEST PHƯƠNG ÁN D: ĐẶT NGƯỠNG MAILER RESPONSIVENESS > 15% HOẶC COUPON
# ==============================================================================
# 1. Ngưỡng cho Mailer
MAILER_THRESHOLD = 0.3
strong_mailer_hhs = set(causal_features[causal_features['Mailer_Responsiveness'] > MAILER_THRESHOLD]['household_key'])

# 2. Tập khách hàng Coupon (toàn thời gian)
coupon_hhs_D = set(obs_txns[obs_txns['COUPON_DISC'] < 0]['household_key'])

# 3. Dùng biến tạm để test
temp_treated_D = df_final['household_key'].apply(
    lambda x: 1 if (x in strong_mailer_hhs) or (x in coupon_hhs_D) else 0
)

print(f"--- KẾT QUẢ PHƯƠNG ÁN D (Mailer > {MAILER_THRESHOLD*100}% hoặc Coupon) ---")
print(temp_treated_D.value_counts(normalize=True) * 100)

--- KẾT QUẢ PHƯƠNG ÁN D (Mailer > 30.0% hoặc Coupon) ---
household_key
1    70.228091
0    29.771909
Name: proportion, dtype: float64
